# Final report

The numbers for the README. Everything is trained on 2024 only; 2025 is the held-out season.

In [1]:
import os
import subprocess
from pathlib import Path

# configs/base.yaml is resolved relative to the repo root
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from f1_strategy_engine.cli import main
from f1_strategy_engine.constants import PointInTimeMode
from f1_strategy_engine.evaluation.leakage import run_leakage
from f1_strategy_engine.evaluation.report import score_models, train_and_score
from f1_strategy_engine.evaluation.splits import race_split
from f1_strategy_engine.features.build import build_modelling_table
from f1_strategy_engine.simulation.parameters import pit_loss_by_circuit, pit_stops

T = pd.Timestamp.now(tz="UTC")
MODE = PointInTimeMode.ROWS_ONLY

commit = subprocess.run(
    ["git", "describe", "--always", "--dirty"], capture_output=True, text=True
).stdout.strip()
print(f"t = {T:%Y-%m-%d %H:%M} UTC | commit = {commit}")

t = 2026-10-11 10:14 UTC | commit = fb32362-dirty


In [2]:
season_2024 = build_modelling_table(T, MODE)
season_2025 = build_modelling_table(T, MODE, seasons=[2025], include_holdout=True)

held_out_races = train_and_score(T, MODE)  # 2024 rounds 1-18 -> 19-24
held_out_season = score_models(season_2024, season_2025)  # all of 2024 -> all of 2025

## 1. Degradation rate per compound

B1's slope, fitted on all of 2024: seconds lost per lap of tyre age.

In [3]:
held_out_season.models["B1"].slopes_.rename("s per lap").round(3).to_frame()

,s per lap
SOFT,0.062
MEDIUM,-0.010
HARD,0.009


## 2. Accuracy on races the model never saw

Error of a lap-time prediction, in seconds. B0 assumes no degradation; B1 adds the per-compound slope.

In [4]:
def accuracy(result):
    overall = result.results.xs("ALL", level="compound")
    b0, b1 = overall.loc["B0", "rmse"], overall.loc["B1", "rmse"]
    return {
        "laps scored": int(overall.loc["B1", "n_scored"]),
        "B0 RMSE (s)": round(b0, 3),
        "B1 RMSE (s)": round(b1, 3),
        "B1 better by (%)": round(100 * (b0 - b1) / b0, 1),
    }


pd.DataFrame.from_dict(
    {
        "2024 races 19-24": accuracy(held_out_races),
        "2025 season": accuracy(held_out_season),
    },
    orient="index",
)

,laps scored,B0 RMSE (s),B1 RMSE (s),B1 better by (%)
2024 races 19-24,4356,1.025,0.998,2.7
2025 season,21384,1.277,1.193,6.6


## 3. Leakage

The same forecasts made with the point-in-time gate on (`ROWS_ONLY`) and off (`NONE`), on 2024 races 19-24.

In [5]:
leakage = run_leakage(T)
leakage.round(3)

,n NONE,RMSE NONE,n ROWS_ONLY,RMSE ROWS_ONLY,inflation %
scored on,,,,,
each arm's own laps,64191,1.095,45557,1.111,1.401
shared laps,45557,1.111,45557,1.111,0.000


In [6]:
own = leakage.loc["each arm's own laps"]
extra = 100 * (own["n NONE"] / own["n ROWS_ONLY"] - 1)
print(f"Without the gate the pipeline forecasts {extra:.0f}% more laps")
print(f"and reports an RMSE {own['inflation %']:.1f}% lower than it really is.")

Without the gate the pipeline forecasts 41% more laps
and reports an RMSE 1.4% lower than it really is.


## 4. Pit loss

Measured from normal (no SC/VSC, slick-to-slick) stops in 2024 rounds 1-18.

In [7]:
train, _ = race_split(season_2024)
stops = pit_stops(train)
print(f"Median pit loss: {stops['loss'].median():.1f} s over {len(stops)} stops")
pit_loss_by_circuit(stops).round(1).rename("pit loss (s)").to_frame()

Median pit loss: 22.4 s over 438 stops


,pit loss (s)
circuit,
Baku,22.3
Barcelona,22.4
Budapest,20.1
Imola,28.0
Jeddah,21.4
Marina Bay,29.9
Melbourne,21.2
Miami,19.6
Monaco,19.3


## 5. Recommendation demo

2024 Spa-Francorchamps, Russell, after lap 25 (on HARD since lap 11).

In [8]:
args = "recommend --season 2024 --round 14 --driver RUS --lap 25"
main(args.split());

Race: 2024 R14 Spa-Francorchamps · Driver RUS · Lap 25 of 44 · On HARD, stint lap 15
Degradation: HARD 0.004 s/lap · pit loss 18.8 s

  Stay out to the end      projected remaining time   2045.9 s
  Box lap 28 -> SOFT       projected remaining time   2056.3 s
  Box lap 27 -> SOFT       projected remaining time   2056.3 s
  Box lap 29 -> SOFT       projected remaining time   2056.4 s
  Box lap 26 -> SOFT       projected remaining time   2056.4 s

Recommendation: Stay out to the end, 10.4 s ahead of Box lap 28 -> SOFT.
NOTE: deterministic, single-car projection in fuel-corrected time. No safety cars, traffic, rivals or tyre cliff.
